<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula%2028b%20-%20MLP%20-%20PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1437, 64)
(360, 64)
(1437,)
(360,)


In [45]:
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim

class ShallowNeuralNetwork(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=200):
    self.max_iter = max_iter
    self.model = None
  def fit(self, X, y):
    input_shape = X.shape[1]
    self.labels, ids = np.unique(y, return_inverse=True)
    output_shape = len(self.labels)

    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(ids, dtype=torch.long)
    self.model = nn.Sequential(
      nn.Linear(input_shape, output_shape),
      nn.Softmax(dim=1)
    )
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(self.model.parameters())
    for epoch in range(self.max_iter):
      optimizer.zero_grad()
      outputs = self.model(X_tensor)
      loss = criterion(outputs, y_tensor)
      loss.backward()
      optimizer.step()

    return self

  def predict_proba(self, X):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
      outputs = self.model(X_tensor)
    return outputs.numpy()

  def predict(self, X):
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    return self.labels[y_pred]

clf = ShallowNeuralNetwork(500)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9611111111111111


In [39]:
class SigleHiddenLayer(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=200, n_hidden_neurons=128):
    self.max_iter = max_iter
    self.n_hidden_neurons = n_hidden_neurons
    self.model = None

  def fit(self, X, y):
    input_shape = X.shape[1]
    self.labels, ids = np.unique(y, return_inverse=True)
    output_shape = len(self.labels)

    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(ids, dtype=torch.long)
    self.model = nn.Sequential(
      nn.Linear(input_shape, self.n_hidden_neurons),
      nn.ReLU(),
      nn.Linear(self.n_hidden_neurons, output_shape),
      nn.Softmax(dim=1)
    )
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(self.model.parameters())
    for epoch in range(self.max_iter):
      optimizer.zero_grad()
      outputs = self.model(X_tensor)
      loss = criterion(outputs, y_tensor)
      loss.backward()
      optimizer.step()

    return self

  def predict_proba(self, X):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
      outputs = self.model(X_tensor)
    return outputs.numpy()

  def predict(self, X):
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    return self.labels[y_pred]

clf = SigleHiddenLayer(500)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9888888888888889


In [49]:
class MultiLayerPerceptron(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=200, n_hidden=[128]):
    self.max_iter = max_iter
    self.n_hidden = n_hidden
    self.model = None

  def fit(self, X, y):
    input_shape = X.shape[1]
    self.labels, ids = np.unique(y, return_inverse=True)
    output_shape = len(self.labels)

    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(ids, dtype=torch.long)

    layers = [nn.Linear(input_shape, self.n_hidden[0]), nn.ReLU()]
    for i in range(len(self.n_hidden)-1):
      layers.append(nn.Linear(self.n_hidden[i], self.n_hidden[i+1]))
      layers.append(nn.ReLU())
    layers.append(nn.Linear(self.n_hidden[-1], output_shape))
    layers.append(nn.Softmax(dim=1))
    self.model = nn.Sequential(*layers)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(self.model.parameters())
    for epoch in range(self.max_iter):
      optimizer.zero_grad()
      outputs = self.model(X_tensor)
      loss = criterion(outputs, y_tensor)
      loss.backward()
      optimizer.step()

    return self

  def predict_proba(self, X):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
      outputs = self.model(X_tensor)
    return outputs.numpy()

  def predict(self, X):
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    return self.labels[y_pred]

clf = MultiLayerPerceptron(1000, n_hidden=[128,128])
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9777777777777777
